# Traduction multilingue Éwé → {Anglais, Français} (NLLB + LoRA)

Au lieu d'entraîner **un modèle par langue cible**, on entraîne **un seul adaptateur**
LoRA capable de traduire l'éwé vers **l'anglais ET le français**.

**Idée clé (NLLB) :** le modèle sait déjà produire plusieurs langues. La langue de sortie
est choisie par un *token de langue* placé au **début des `labels`** (et, à l'inférence,
via `forced_bos_token_id`). Il suffit donc de mélanger, dans le même corpus d'entraînement :

- les exemples `ewe_Latn → eng_Latn` (labels préfixés du token `eng_Latn`)
- les exemples `ewe_Latn → fra_Latn` (labels préfixés du token `fra_Latn`)

On obtient un seul modèle, plus compact à déployer, qui partage ses connaissances
entre les deux directions (utile pour une langue peu dotée comme l'éwé).


In [ ]:
!pip install -q evaluate sacrebleu

In [ ]:
import json
import os
import re
from pathlib import Path

# Un seul GPU (Kaggle T4 x2 -> DataParallel sature la VRAM sinon)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import numpy as np
import evaluate
from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass  # local : huggingface-cli login ou variable d'environnement HF_TOKEN

ds_raw = load_dataset(
    "romaricnadjire/ewe-nllb-translation",
    data_files={
        "train":      "train.jsonl",
        "validation": "validation.jsonl",
        "test":       "test.jsonl",
    },
    token=True,
)
ds_raw


## 1. Configuration

`TGT_LANGS` contient la **liste** des langues cibles. Pour en ajouter une autre prise en
charge par NLLB (ex. `yor_Latn`), il suffit de l'ajouter ici.

In [ ]:
MODEL_NAME   = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR   = "./output/nllb-ewe-multi"
ADAPTER_DIR  = "./output/nllb-ewe-multi/adapter"
RESULTS_FILE = "./output/resultats_multi.json"

SRC_LANG  = "ewe_Latn"                 # source unique
TGT_LANGS = ["eng_Latn", "fra_Latn"]   # cibles multiples

MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

LEARNING_RATE    = 3e-4
BATCH_SIZE_TRAIN = 4
BATCH_SIZE_EVAL  = 8
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS       = 3
WARMUP_RATIO     = 0.06
WEIGHT_DECAY     = 0.01

LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration OK")
print(f"  Source : {SRC_LANG}")
print(f"  Cibles : {TGT_LANGS}")
print(f"  Batch effectif : {BATCH_SIZE_TRAIN} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE_TRAIN * GRAD_ACCUM_STEPS}")


## 2. Nettoyage (par paire de langues)

`clean_pair` valide **une** paire (source, cible). On l'appliquera séparément pour chaque
langue cible. On retire : les paires vides, les copies (source == cible), les références
bibliques seules, et les caractères éwé qui « fuiteraient » dans une cible non-éwé.

In [ ]:
EWE_CHARS = set("ŋɖɔɛʋƒãẽĩõũ")
BIBLE_REF_RE = re.compile(r"^\s*\d{1,3}:\d{1,3}(?:-\d{1,3})?\s*$")

def clean_pair(src, tgt, tgt_lang):
    # Retourne True si la paire (src, tgt) est exploitable.
    src = (src or "").strip()
    tgt = (tgt or "").strip()
    if not src or not tgt:
        return False
    if src == tgt:                       # copie, pas une traduction
        return False
    if BIBLE_REF_RE.match(src) and src == tgt:
        return False
    if tgt_lang != "ewe_Latn" and sum(c in EWE_CHARS for c in tgt) >= 2:
        return False
    return True

# Apercu : combien de paires propres par langue ?
for lang in TGT_LANGS:
    n = sum(
        clean_pair(ex["translation"].get(SRC_LANG), ex["translation"].get(lang), lang)
        for ex in ds_raw["train"]
    )
    print(f"  train {SRC_LANG} -> {lang} : {n} paires propres")


## 3. Tokenisation multilingue

C'est le coeur du notebook. Pour **chaque** langue cible :

1. on règle `tokenizer.src_lang` et `tokenizer.tgt_lang` ;
2. `text_target=` fait que le tokenizer **préfixe automatiquement** les `labels` avec le
   token de la langue cible (`eng_Latn` ou `fra_Latn`) — c'est ce token qui dira au modèle
   quelle langue produire ;
3. on concatène les sous-ensembles tokenisés, puis on **mélange** (`shuffle`) pour que
   chaque batch contienne un mix des deux langues.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_preprocess(tgt_lang):
    def preprocess(batch):
        tokenizer.src_lang = SRC_LANG
        tokenizer.tgt_lang = tgt_lang          # -> token de langue ajoute aux labels
        sources = [ex.get(SRC_LANG) or "" for ex in batch["translation"]]
        targets = [ex.get(tgt_lang) or "" for ex in batch["translation"]]
        model_inputs = tokenizer(
            sources, text_target=targets, max_length=MAX_INPUT_LEN, truncation=True
        )
        model_inputs["labels"] = [ids[:MAX_TARGET_LEN] for ids in model_inputs["labels"]]
        return model_inputs
    return preprocess

parts = {"train": [], "validation": [], "test": []}
for tgt_lang in TGT_LANGS:
    sub = ds_raw.filter(
        lambda ex: clean_pair(
            ex["translation"].get(SRC_LANG), ex["translation"].get(tgt_lang), tgt_lang
        )
    )
    tok = sub.map(
        make_preprocess(tgt_lang),
        batched=True,
        remove_columns=ds_raw["train"].column_names,
        desc=f"Tokenisation {tgt_lang}",
    )
    for split in parts:
        parts[split].append(tok[split])

tokenized = DatasetDict({
    split: concatenate_datasets(p).shuffle(seed=42) for split, p in parts.items()
})
print(tokenized)


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)


## 4. LoRA + modèle

Identique au notebook mono-langue : on gèle NLLB et on n'entraîne que les petites matrices
LoRA sur `q_proj` / `v_proj`.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

peft_config = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = LORA_TARGET_MODULES,
    bias           = "none",
    use_rslora     = True,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## 5. Entraînement

**Différence importante avec le notebook mono-langue :** pendant l'entraînement, le jeu de
validation mélange l'anglais et le français. Or `predict_with_generate` ne peut forcer
qu'**une** langue de sortie à la fois — calculer un BLEU « mélangé » n'aurait pas de sens.
On suit donc la **perte de validation** (`eval_loss`) pendant l'entraînement, puis on
calcule BLEU / chrF++ **séparément par langue** à la fin (section 6).

In [ ]:
eval_subset = tokenized["validation"].select(range(min(800, len(tokenized["validation"]))))

training_args = Seq2SeqTrainingArguments(
    output_dir = OUTPUT_DIR,
    num_train_epochs = NUM_EPOCHS,
    eval_steps    = 1000,
    save_steps    = 1000,
    logging_steps = 50,
    per_device_train_batch_size = BATCH_SIZE_TRAIN,
    per_device_eval_batch_size  = BATCH_SIZE_EVAL,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    gradient_checkpointing = True,
    learning_rate     = LEARNING_RATE,
    warmup_ratio      = WARMUP_RATIO,
    lr_scheduler_type = "cosine",
    weight_decay      = WEIGHT_DECAY,
    fp16 = (device == "cuda"),
    # Pas de generation pendant l'eval (cibles melangees) -> on suit eval_loss
    predict_with_generate = False,
    eval_strategy          = "steps",
    save_strategy          = "steps",
    load_best_model_at_end = True,
    metric_for_best_model  = "eval_loss",
    greater_is_better      = False,
    save_total_limit       = 2,
    disable_tqdm           = True,
    logging_first_step     = True,
    report_to              = "none",
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized["train"],
    eval_dataset     = eval_subset,
    processing_class = tokenizer,
    data_collator    = data_collator,
)
print("Trainer multilingue pret.")


In [ ]:
# Reprise automatique depuis le dernier checkpoint (utile sur Kaggle en arriere-plan)
last_ckpt = None
output_path = Path(OUTPUT_DIR)
if output_path.is_dir():
    ckpts = sorted(
        [d for d in output_path.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"Reprise depuis : {last_ckpt}")
    else:
        print("Aucun checkpoint -> entrainement from scratch.")
else:
    print("Aucun checkpoint -> entrainement from scratch.")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\nAdaptateur multilingue sauvegarde : {ADAPTER_DIR}")
print(f"Loss train finale : {train_result.training_loss:.4f}")


## 6. Évaluation finale — par langue

On recharge le meilleur modèle (LoRA fusionné), puis on évalue **chaque** direction
séparément en forçant la bonne langue de sortie via `forced_bos_token_id`.

In [ ]:
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric      = evaluate.load("chrf")

print("Chargement du modele fine-tune...")
base_model_ft = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_ft = PeftModel.from_pretrained(base_model_ft, ADAPTER_DIR).merge_and_unload().to(device)
model_ft.eval()
tokenizer_ft = AutoTokenizer.from_pretrained(ADAPTER_DIR)

def translate_batch(model, tok, sources, src_lang, tgt_lang, batch_size=16, max_new_tokens=128):
    tok.src_lang = src_lang
    forced_bos = tok.convert_tokens_to_ids(tgt_lang)
    preds = []
    for i in range(0, len(sources), batch_size):
        batch = sources[i:i + batch_size]
        inputs = tok(batch, return_tensors="pt", padding=True, truncation=True,
                     max_length=MAX_INPUT_LEN).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                 max_new_tokens=max_new_tokens, num_beams=4)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds

def eval_lang(split, tgt_lang, n=None):
    pairs = [
        (ex["translation"].get(SRC_LANG), ex["translation"].get(tgt_lang))
        for ex in split
        if clean_pair(ex["translation"].get(SRC_LANG), ex["translation"].get(tgt_lang), tgt_lang)
    ]
    if n:
        pairs = pairs[:n]
    sources = [p[0] for p in pairs]
    refs    = [p[1] for p in pairs]
    preds   = translate_batch(model_ft, tokenizer_ft, sources, SRC_LANG, tgt_lang)
    bleu = sacrebleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    chrf = chrf_metric.compute(predictions=preds, references=[[r] for r in refs], word_order=2)
    return {"n": len(sources), "bleu": round(bleu["score"], 2), "chrf++": round(chrf["score"], 2)}

results = {}
for lang in TGT_LANGS:
    print(f"=== test : {SRC_LANG} -> {lang} ===")
    results[lang] = eval_lang(ds_raw["test"], lang)
    print(f"  BLEU={results[lang]['bleu']}  chrF++={results[lang]['chrf++']}  (n={results[lang]['n']})")

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\nResultats sauvegardes : {RESULTS_FILE}")


In [ ]:
# Exemples : la meme phrase eve traduite vers les deux langues
sample   = ds_raw["test"].select(range(4))
sources  = [ex["translation"][SRC_LANG] for ex in sample]
for lang in TGT_LANGS:
    preds = translate_batch(model_ft, tokenizer_ft, sources, SRC_LANG, lang)
    print(f"\n----- {SRC_LANG} -> {lang} -----")
    for s, p in zip(sources, preds):
        print(f"  EVE : {s}")
        print(f"  ->  : {p}\n")
